# Text Classification — the TF-IDF baseline

The workhorse NLP task, end to end: build a corpus, compare representations, compare
classifiers, tune regularization, then actually look at the errors.

**Runs offline in seconds** — no downloads. The corpus is synthetic *on purpose*: the three
classes share a vocabulary and differ only by emphasis, plus 5% label noise. Disjoint
vocabularies would score 100% and teach nothing.

Concepts: [`01_Text_Representation.md`](01_Text_Representation.md) ·
Tasks and metrics: [`02_Tasks_and_Evaluation.md`](02_Tasks_and_Evaluation.md)

In [ ]:
import numpy as np
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import classification_report, confusion_matrix


## 1. A corpus with genuine class overlap

Each class emphasises its own theme words (weight 3.0) but still draws the *other* classes'
theme words (weight 0.7) and a shared generic pool (1.0). That overlap plus label noise is
what makes the task non-trivial.

In [ ]:
THEMES = {
    "space":    "orbit launch rocket satellite mission payload telescope lunar crew module".split(),
    "medicine": "patient clinical trial dosage symptom diagnosis treatment therapy chronic vaccine".split(),
    "finance":  "market equity portfolio dividend yield inflation trading asset liquidity valuation".split(),
}
GENERIC = """system model risk value analysis result report study data increase decrease signal
rate level test sample control growth failure cost design limit target range""".split()
FILLER = "the a of and to in for with on is are was were this that it as at by from".split()
CLASSES = list(THEMES)
CONTENT = [w for ws in THEMES.values() for w in ws] + GENERIC


In [ ]:
def make_corpus(n_per_class=240, seed=0, theme_w=3.0, cross_w=0.7,
                doc_len=(14, 32), label_noise=0.05):
    rng = np.random.default_rng(seed)
    idx = {w: i for i, w in enumerate(CONTENT)}
    dists = []
    for cls in CLASSES:
        w = np.full(len(CONTENT), 1.0)                       # generic pool: baseline
        for t in THEMES[cls]:
            w[idx[t]] = theme_w                              # own theme: emphasised
        for other in CLASSES:
            if other != cls:
                for t in THEMES[other]:
                    w[idx[t]] = cross_w                      # other themes: still possible
        w *= rng.uniform(0.8, 1.2, len(w))
        dists.append(w / w.sum())

    docs, labels = [], []
    for ci, _ in enumerate(CLASSES):
        for _ in range(n_per_class):
            L = rng.integers(*doc_len)
            k = int(L * 0.55)
            toks = list(rng.choice(CONTENT, k, p=dists[ci])) + list(rng.choice(FILLER, L - k))
            rng.shuffle(toks)
            docs.append(" ".join(toks))
            labels.append(ci)

    labels = np.array(labels)
    flip = rng.random(len(labels)) < label_noise
    labels[flip] = rng.integers(0, len(CLASSES), flip.sum())
    return np.array(docs), np.array([CLASSES[i] for i in labels])


In [ ]:
X, y = make_corpus()
print("corpus:", len(X), "docs |", dict(zip(*np.unique(y, return_counts=True))))
print("example:", X[0])


## 2. Does the representation matter?

Raw counts vs TF-IDF vs n-grams vs stopword removal. Watch the count→TF-IDF jump: that is
IDF down-weighting words frequent in *every* class.

Note n-grams multiply the feature count — check whether the accuracy pays for it.

In [ ]:
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.25, random_state=0, stratify=y)
print("train", len(Xtr), "test", len(Xte))


## 3. Does the classifier matter?

Far less than the representation, usually. MultinomialNB is a strong, very fast baseline on
count-like features; LinearSVC and LogisticRegression are typically within a point of each
other, with LogisticRegression giving you probabilities.

In [ ]:
print("\n=== representation comparison (LogisticRegression) ===")
configs = {
    "Count  unigram":     CountVectorizer(),
    "TF-IDF unigram":     TfidfVectorizer(),
    "TF-IDF 1-2gram":     TfidfVectorizer(ngram_range=(1, 2), min_df=2),
    "TF-IDF no stopword": TfidfVectorizer(stop_words="english"),
    "TF-IDF sublinear":   TfidfVectorizer(sublinear_tf=True),
}
print(f"{'vectorizer':20s} {'feats':>6s} {'CV':>7s} {'test':>7s}")
for name, vec in configs.items():
    pipe = make_pipeline(vec, LogisticRegression(max_iter=2000))
    cv = cross_val_score(pipe, Xtr, ytr, cv=5).mean()
    pipe.fit(Xtr, ytr)
    print(f"{name:20s} {len(pipe[0].vocabulary_):6d} {cv:7.3f} {pipe.score(Xte, yte):7.3f}")


## 4. Regularization

`C` is the *inverse* regularization strength, so large `C` = weak penalty. With |V| features
and few hundred documents this is the `p ≫ n` regime, where the penalty is doing real work —
watch train accuracy climb while test accuracy falls.

In [ ]:
print("\n=== classifier comparison (TF-IDF unigram) ===")
for clf in [LogisticRegression(max_iter=2000), LinearSVC(), MultinomialNB()]:
    pipe = make_pipeline(TfidfVectorizer(), clf).fit(Xtr, ytr)
    print(f"  {type(clf).__name__:20s} {pipe.score(Xte, yte):.3f}")


## 5. The full report

In [ ]:
print("\n=== regularization sweep ===")
print(f"  {'C':>6s} {'train':>7s} {'test':>7s}")
for C in [0.01, 0.1, 1, 10, 100]:
    pipe = make_pipeline(TfidfVectorizer(), LogisticRegression(C=C, max_iter=2000)).fit(Xtr, ytr)
    print(f"  {C:6g} {pipe.score(Xtr, ytr):7.3f} {pipe.score(Xte, yte):7.3f}")


## 6. What did the model actually learn?

For a linear model on TF-IDF the coefficients *are* the explanation — one weight per word per
class. This interpretability is a real reason to keep the baseline around, and the fastest way
to catch leakage: if a top feature is an artifact (a header, a source marker, boilerplate),
you've found a bug, not a signal.

In [ ]:
pipe = make_pipeline(TfidfVectorizer(), LogisticRegression(max_iter=2000)).fit(Xtr, ytr)
pred = pipe.predict(Xte)
print("\n", classification_report(yte, pred, digits=3))
print("confusion (rows=true):", CLASSES)
print(confusion_matrix(yte, pred, labels=CLASSES))


## 7. Error analysis

Two piles worth separating:

- **Confident mistakes** — high probability, wrong answer. Often mislabeled data (here, the
  injected 5% noise), and the first place to look for annotation problems.
- **Low-confidence predictions** — the genuinely ambiguous documents. These tell you where the
  classes actually overlap, and whether an abstain/route-to-human tier would pay for itself.

In [ ]:
vec, clf = pipe[0], pipe[1]
terms = np.array(vec.get_feature_names_out())
print("\n=== most informative features per class ===")
for i, cls in enumerate(clf.classes_):
    top = np.argsort(clf.coef_[i])[-8:][::-1]
    on = sum(t in THEMES[cls] for t in terms[top])
    print(f"  {cls:9s} [{on}/8 on-theme] {', '.join(terms[top])}")


## Takeaways

1. TF-IDF beats raw counts because IDF suppresses words frequent everywhere.
2. n-grams cost features fast — make them prove their worth.
3. The representation matters more than the classifier.
4. The coefficients are free interpretability; use them to hunt leakage.
5. Confident mistakes ≈ label noise; unconfident ones ≈ real ambiguity.

**Try next:** swap the synthetic corpus for a real one —
`from sklearn.datasets import fetch_20newsgroups` (downloads ~14 MB on first use) — and rerun.
The relative ordering of the configurations should hold.